TRAINING A PRETRAINED MODEL FOR INTENT CLASSIFICATION

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("helper_docs/intent_examples.csv")
df.head()

,text,intent
0,"Can I get loan options for 50,000 TL for 12 mo...",Request_Loan_Proposals
1,"Show me loans for 100,000 TL with a 5-year term.",Request_Loan_Proposals
2,"I need a loan proposal for 200,000 TL for 36 m...",Request_Loan_Proposals
3,"Find me loan rates for 75,000 TL over 24 months.",Request_Loan_Proposals
4,"I'm looking for loans up to 150,000 TL for a 1...",Request_Loan_Proposals


In [3]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['intent'] = label_encoder.fit_transform(df['intent'])
df


,text,intent
0,"Can I get loan options for 50,000 TL for 12 mo...",3
1,"Show me loans for 100,000 TL with a 5-year term.",3
2,"I need a loan proposal for 200,000 TL for 36 m...",3
3,"Find me loan rates for 75,000 TL over 24 months.",3
4,"I'm looking for loans up to 150,000 TL for a 1...",3
...,...,...
275,How do I calculate the break cost on a fixed-r...,0
276,"What is a bridging loan, and how does it work?",0
277,Can I get a loan for agricultural purposes?,0
278,What are the requirements for a student loan?,0


In [4]:
import joblib

encoder_save_path = 'helper_docs/label_encoder.joblib'
joblib.dump(label_encoder, encoder_save_path)

['helper_docs/label_encoder.joblib']

In [5]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.1)

In [6]:
from transformers import DistilBertTokenizer
import torch
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def encode_examples(df, max_length=512):
    input_ids = []
    attention_masks = []
    labels = []

    for _, row in df.iterrows():
        encoded_dict = tokenizer.encode_plus(
            row['text'],                     
            add_special_tokens = True,
            max_length = max_length,
            pad_to_max_length = True,
            return_attention_mask = True,
            return_tensors = 'pt',
        )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])
        labels.append(row['intent'])

    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(labels)

    return input_ids, attention_masks, labels

train_inputs, train_masks, train_labels = encode_examples(train_df)
validation_inputs, validation_masks, validation_labels = encode_examples(val_df)


/Users/okanyenigun/Desktop/codes/projects/genai_work/notebooks/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/Users/okanyenigun/Desktop/codes/projects/genai_work/notebooks/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:2618: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the

In [7]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

batch_size = 32 

train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

validation_data = TensorDataset(validation_inputs, validation_masks, validation_labels)
validation_sampler = SequentialSampler(validation_data)
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)


In [8]:
from transformers import DistilBertForSequenceClassification, AdamW

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels = len(df['intent'].unique()),
    output_attentions = False,
    output_hidden_states = False,
)

optimizer = AdamW(model.parameters(), lr=2e-5)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'pre_classifier.bias', 'pre_classifier.weight', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/okanyenigun/Desktop/codes/projects/genai_work/notebooks/venv/lib/python3.11/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [11]:
import torch
from transformers import get_linear_schedule_with_warmup
import numpy as np
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 4

total_steps = len(train_dataloader) * epochs

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)


/Users/okanyenigun/Desktop/codes/projects/genai_work/notebooks/venv/lib/python3.11/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [12]:

def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

for epoch in range(0, epochs):
    print('======== Epoch {:} / {:} ========'.format(epoch + 1, epochs))
    
    model.train()
    
    total_train_loss = 0
    total_train_accuracy = 0

    for step, batch in enumerate(train_dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        
        model.zero_grad()
        
        outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
        loss = outputs.loss
        logits = outputs.logits
        
        total_train_loss += loss.item()
        
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        scheduler.step()

        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        total_train_accuracy += flat_accuracy(logits, label_ids)

    avg_train_accuracy = total_train_accuracy / len(train_dataloader)
    avg_train_loss = total_train_loss / len(train_dataloader)
    
    print("  Average training accuracy: {0:.2f}".format(avg_train_accuracy))
    print("  Average training loss: {0:.2f}".format(avg_train_loss))

    # validation
    
    model.eval()

    total_eval_accuracy = 0
    total_eval_loss = 0

    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        
        with torch.no_grad():
            outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
        
        loss = outputs.loss
        logits = outputs.logits
        
        total_eval_loss += loss.item()

        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        total_eval_accuracy += flat_accuracy(logits, label_ids)

    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    avg_val_loss = total_eval_loss / len(validation_dataloader)
    
    print("  Average validation accuracy: {0:.2f}".format(avg_val_accuracy))
    print("  Average validation loss: {0:.2f}".format(avg_val_loss))


======== Epoch 1 / 4 ========
  Average training accuracy: 0.59
  Average training loss: 1.13
  Average validation accuracy: 0.64
  Average validation loss: 0.99
======== Epoch 2 / 4 ========
  Average training accuracy: 0.69
  Average training loss: 0.97
  Average validation accuracy: 0.79
  Average validation loss: 0.86
======== Epoch 3 / 4 ========
  Average training accuracy: 0.81
  Average training loss: 0.86
  Average validation accuracy: 0.82
  Average validation loss: 0.79
======== Epoch 4 / 4 ========
  Average training accuracy: 0.83
  Average training loss: 0.80
  Average validation accuracy: 0.82
  Average validation loss: 0.76


In [13]:

def get_validation_predictions(model, dataloader):
    model.eval()
    predictions , true_labels = [], []
    
    for batch in dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        
        with torch.no_grad():
            outputs = model(b_input_ids, attention_mask=b_input_mask)
        
        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        
        predictions.append(logits)
        true_labels.append(label_ids)
    
    return predictions, true_labels

predictions, true_labels = get_validation_predictions(model, validation_dataloader)

flat_predictions = np.concatenate(predictions, axis=0)
flat_predictions = np.argmax(flat_predictions, axis=1).flatten()
flat_true_labels = np.concatenate(true_labels, axis=0)

print(classification_report(flat_true_labels, flat_predictions, target_names=label_encoder.classes_))


                        precision    recall  f1-score   support

       General_Inquiry       1.00      0.90      0.95        10
  Inquire_Loan_Details       0.75      0.67      0.71         9
     Proceed_With_Loan       0.50      0.67      0.57         3
Request_Loan_Proposals       0.86      1.00      0.92         6

              accuracy                           0.82        28
             macro avg       0.78      0.81      0.79        28
          weighted avg       0.84      0.82      0.82        28



In [14]:
# save
model_save_path = 'trained_model'

import os
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)

model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)


('trained_model/tokenizer_config.json',
 'trained_model/special_tokens_map.json',
 'trained_model/vocab.txt',
 'trained_model/added_tokens.json')

In [15]:
# load

model = DistilBertForSequenceClassification.from_pretrained(model_save_path)
tokenizer = DistilBertTokenizer.from_pretrained(model_save_path)

model.eval()


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [16]:
def predict(text, model, tokenizer):
    inputs = tokenizer.encode_plus(
        text,
        None,
        add_special_tokens=True,
        max_length=512,
        pad_to_max_length=True,
        return_attention_mask=True,
        return_tensors='pt',
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    logits = logits.detach().cpu().numpy()

    probabilities = torch.nn.functional.softmax(torch.tensor(logits), dim=1).numpy()

    predicted_class_idx = np.argmax(probabilities, axis=1).flatten()

    predicted_class = label_encoder.inverse_transform(predicted_class_idx)[0]
    return predicted_class

example_text = "I want to know more about a loan for 200,000 TL."
prediction = predict(example_text, model, tokenizer)
print(f"Predicted intent: {prediction}")


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


/Users/okanyenigun/Desktop/codes/projects/genai_work/notebooks/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:2618: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Predicted intent: Request_Loan_Proposals
